In [1]:
import os
import re
import glob
import numpy as np
import pandas as pd

def create_split_file(
    folder_path,
    output_csv="train_test_split.csv",
    test_ratio=0.2,
    random_state=42
):

    np.random.seed(random_state)
    records = []

    file_list = sorted(glob.glob(os.path.join(folder_path, "*.csv")))

    for file in file_list:
        filename = os.path.basename(file)
        match = re.match(r"(\d+)-(\d+)_(\d+)\.csv", filename)

        if match is None:
            continue

        richness = int(match.group(1))
        n_rows = len(pd.read_csv(file))
        idx = np.random.permutation(n_rows)
        n_test = int(n_rows * test_ratio)
        idx_te = idx[:n_test]
        idx_tr = idx[n_test:]

        for row_idx in idx_tr:
            records.append({
                "file": filename,
                "row": row_idx,
                "richness": richness,
                "split": "tr"
            })

        for row_idx in idx_te:
            records.append({
                "file": filename,
                "row": row_idx,
                "richness": richness,
                "split": "te"
            })

        print(
            f"{filename}: "
            f"train={len(idx_tr)}, "
            f"test={len(idx_te)}"
        )

    split_df = pd.DataFrame(records)
    split_df.to_csv(output_csv, index=False)
    print(f"Saved split file: {output_csv}")
    return split_df

def CreateTrainTestSplit_by_Property(data_dir, sort_col, output_csv="split_MD.csv",
                         richness_col="richness", ascending=False, train_frac=0.8):
    '''
    Record train-test splits for every CSV file in data_dir.

    Parameters
    ----------
    data_dir : str
        Folder containing CSV files.
    sort_col : str
        Column used to sort samples within each file.
    output_csv : str
        Output CSV path.
    richness_col : str
        Column containing richness values.
    ascending : bool
        True for ascending order, False for descending order.
    train_frac : float
        Fraction of samples assigned to the training set.

    Returns
    -------
    split_df : DataFrame
        Columns: file, row, richness, split.
    '''
    if not 0 < train_frac < 1:
        raise ValueError("train_frac must be between 0 and 1.")

    csvfiles = sorted([
        f for f in os.listdir(data_dir)
        if f.endswith(".csv")
    ])

    records = []
    output_path = os.path.abspath(output_csv)

    for file in csvfiles:
        filepath = os.path.join(data_dir, file)

        # Avoid reading the output CSV if it is inside data_dir
        if os.path.abspath(filepath) == output_path:
            continue

        df = pd.read_csv(filepath)

        if df.empty:
            print(f"Skip empty file: {file}")
            continue

        # Check required columns
        missing_cols = [col for col in [richness_col, sort_col]
                        if col not in df.columns]
        if missing_cols:
            raise KeyError(f"{file} is missing columns: {missing_cols}")

        # Richness is constant within each file
        richness = df[richness_col].iloc[0]

        # Sort samples while preserving their original row indices
        sorted_rows = df.sort_values(
            by=sort_col, ascending=ascending, kind="mergesort"
        ).index.to_numpy()

        # First train_frac samples for training, remaining for testing
        n_train = int(len(df) * train_frac)
        splits = ["tr"] * n_train + ["te"] * (len(df) - n_train)

        records.append(pd.DataFrame({
            "file": os.path.splitext(file)[0],
            "row": sorted_rows,
            "richness": richness,
            "split": splits
        }))

        print(f"{file}: total={len(df)}, tr={n_train}, te={len(df)-n_train}")

    # Combine all files and save
    columns = ["file", "row", "richness", "split"]
    split_df = pd.concat(records, ignore_index=True) if records else pd.DataFrame(columns=columns)

    split_df.to_csv(output_csv, index=False)
    print(f"Saved: {output_csv}")
    print(f"Total samples: {len(split_df)}")
    print(split_df["split"].value_counts())

    return split_df

def SubsampleTrainTestSplit(input_csv="split_MD.csv",
                            output_csv="split_MD_seed42.csv",
                            sample_frac=0.9,
                            random_state=42):
    '''
    Randomly subsample existing train/test splits.

    Randomly select sample_frac of the existing training samples (tr)
    and testing samples (te), respectively.

    Parameters
    ----------
    input_csv : str
        Existing train-test split record CSV.
    output_csv : str
        Output CSV for the subsampled split records.
    sample_frac : float
        Fraction of samples retained within each split.
    random_state : int
        Random seed for reproducibility.

    Returns
    -------
    split_df : DataFrame
        Subsampled split records.
    '''
    if not 0 < sample_frac <= 1:
        raise ValueError("sample_frac must be in (0, 1].")

    df = pd.read_csv(input_csv)
    required_cols = ["file", "row", "richness", "split"]

    if not all(col in df.columns for col in required_cols):
        raise ValueError(f"Input CSV must contain columns: {required_cols}")

    if df["split"].isna().any() or not set(df["split"].unique()).issubset({"tr", "te"}):
        raise ValueError("The split column must contain only 'tr' and 'te'.")

    rng = np.random.default_rng(random_state)
    selected = []

    for split_name in ["tr", "te"]:
        sub_df = df[df["split"] == split_name]
        n = len(sub_df)
        n_keep = min(n, max(1, int(n * sample_frac))) if n > 0 else 0

        # Randomly select samples without replacement
        idx = rng.choice(n, size=n_keep, replace=False)
        selected.append(sub_df.iloc[idx])

        print(f"{split_name}: {n} -> {n_keep}")

    split_df = pd.concat(selected, ignore_index=True)

    # Sort for easier inspection; this does not change split membership
    split_df = split_df.sort_values(
        ["file", "row"], kind="mergesort"
    ).reset_index(drop=True)

    os.makedirs(os.path.dirname(os.path.abspath(output_csv)), exist_ok=True)
    split_df.to_csv(output_csv, index=False)

    print(f"Saved: {output_csv}")
    print(f"Total samples: {len(split_df)}")
    print(split_df["split"].value_counts())

    return split_df


**Generate a single tr-te split strategy**

In [ ]:
# create_split_file("/work/home/acychkqsp6/zrm/MasterThesis/Databases/Ver1.9/ver1.9.6_upto20_fl", output_csv="split_fl_1.csv", test_ratio=0.2, random_state=42)
# create_split_file("../../Databases/Ver1.9/ver1.9.6_upto20", output_csv="split_MD_1.csv", test_ratio=0.2, random_state=33)
# create_split_file("../../Databases/Ver1.9/ver1.9.6_upto20", output_csv="split_MD_2.csv", test_ratio=0.2, random_state=66)
create_split_file("../../Databases/Ver1.9/ver1.9.6_upto20", output_csv="split_MD_3.csv", test_ratio=0.2, random_state=99)


**Batch generate tr-te split strategies**

In [ ]:
for rep in range(1,4):
    create_split_file(
        "../../Databases/Ver1.9/ver1.9.6.1_upto20", 
        output_csv=f"split_ver1.9.6.1_upto20/split_{rep}.csv", 
        test_ratio=0.2, random_state=rep)


**Initial sequential tr-te split**

In [3]:
rootdir = "../../Databases/Ver2.0/ver2.0_upto20"

split_df = CreateTrainTestSplit_by_Property(
    data_dir=rootdir,
    sort_col="gamma_neg_mean",  # Column name to sort by
    output_csv="SortByProperty/negmean_ascend.csv",
    richness_col="richness",
    ascending=True,
    train_frac=0.8
)


10-20_97668.csv: total=97668, tr=78134, te=19534
11-20_97285.csv: total=97285, tr=77828, te=19457
12-20_96851.csv: total=96851, tr=77480, te=19371
13-20_96409.csv: total=96409, tr=77127, te=19282
14-20_95806.csv: total=95806, tr=76644, te=19162
15-20_95532.csv: total=95532, tr=76425, te=19107
16-20_95044.csv: total=95044, tr=76035, te=19009
17-20_94660.csv: total=94660, tr=75728, te=18932
18-20_94491.csv: total=94491, tr=75592, te=18899
19-20_94133.csv: total=94133, tr=75306, te=18827
20-20_93696.csv: total=93696, tr=74956, te=18740
3-20_101539.csv: total=101539, tr=81231, te=20308
4-20_100823.csv: total=100823, tr=80658, te=20165
5-20_100334.csv: total=100334, tr=80267, te=20067
6-20_99645.csv: total=99645, tr=79716, te=19929
7-20_99142.csv: total=99142, tr=79313, te=19829
8-20_98513.csv: total=98513, tr=78810, te=19703
9-20_98090.csv: total=98090, tr=78472, te=19618
Saved: SortByProperty/negmean_ascend.csv
Total samples: 1749661
split
tr    1399722
te     349939
Name: count, dtype: i

**After initial sequential split, randomly subsample as tr-te**

In [4]:
for rep in range(1,4):
    split_df = SubsampleTrainTestSplit(
        input_csv="SortByProperty/negmean_ascend.csv",
        output_csv=f"split_negmean_ascend/split_{rep}.csv",
        sample_frac=0.9,
        random_state=rep
    )


tr: 1399722 -> 1259749
te: 349939 -> 314945
Saved: split_negmean_ascend/split_1.csv
Total samples: 1574694
split
tr    1259749
te     314945
Name: count, dtype: int64
tr: 1399722 -> 1259749
te: 349939 -> 314945
Saved: split_negmean_ascend/split_2.csv
Total samples: 1574694
split
tr    1259749
te     314945
Name: count, dtype: int64
tr: 1399722 -> 1259749
te: 349939 -> 314945
Saved: split_negmean_ascend/split_3.csv
Total samples: 1574694
split
tr    1259749
te     314945
Name: count, dtype: int64
